# Feeding data: Dataset and DataLoader

_PyTorch (a complete course)_

**Wrap your data in a Dataset, hand it to a DataLoader, and get batched, shuffled, augmented, parallel-loaded tensors that keep the GPU (Graphics Processing Unit) busy.**

Separate what one sample is from how samples become batches. The
       Dataset owns the first; the DataLoader owns the second. Because of that clean
       split, the same DataLoader machinery &mdash; batching, shuffling, parallel workers, pinned memory &mdash;
       works for images, text, audio, or numpy tables without change. You write a tiny Dataset; you get an
       industrial data pipeline.

---

This notebook is a practice scaffold for the **Feeding data: Dataset and DataLoader** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)          # reproducible shuffling / results
device = "cuda" if torch.cuda.is_available() else "cpu"

# ======================================================================
# 1) A CUSTOM Dataset over numpy arrays  (the most common starting point)
# ======================================================================
class ArrayDataset(Dataset):
    def __init__(self, X, y):
        # keep the raw arrays; convert per-sample in __getitem__
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)

    def __len__(self):                      # how many samples
        return len(self.X)

    def __getitem__(self, i):               # ONE (x, y) pair, correct dtypes
        x = torch.from_numpy(self.X[i])                      # float32 feature vector
        y = torch.tensor(self.y[i], dtype=torch.long)        # long class index (CrossEntropyLoss wants this)
        return x, y

# synthetic 2-class data: 500 samples, 3 features
X = np.random.randn(500, 3).astype(np.float32)
y = (X[:, 0] + X[:, 1] > 0).astype(np.int64)
ds = ArrayDataset(X, y)
print("dataset size:", len(ds), "| one sample:", ds[0][0].shape, ds[0][1].item())

# The DataLoader: batch + shuffle (TRAIN only) + parallel workers + pinned memory for GPU.
loader = DataLoader(
    ds,
    batch_size=64,
    shuffle=True,                 # shuffle the TRAIN loader only
    num_workers=2,                # parallel worker processes prefetch batches (0 on Windows/notebooks)
    pin_memory=(device == "cuda") # faster CPU->GPU copy when training on GPU
    , drop_last=False             # keep the final short batch
)

xb, yb = next(iter(loader))       # grab ONE batch
print("batch x:", xb.shape, xb.dtype, "| batch y:", yb.shape, yb.dtype)
# batch x: torch.Size([64, 3]) torch.float32 | batch y: torch.Size([64]) torch.int64

# ======================================================================
# 2) TORCHVISION MNIST with transforms.Compose + a DataLoader
# ======================================================================
from torchvision import datasets, transforms

# TRAIN transform: augment, then ToTensor (-> [0,1]), then Normalize (MNIST mean/std).
train_tf = transforms.Compose([
    transforms.RandomCrop(28, padding=2),      # augmentation: TRAIN ONLY
    transforms.ToTensor(),                     # PIL/np uint8 [0,255] -> float tensor [0,1], shape (1,28,28)
    transforms.Normalize((0.1307,), (0.3081,)),# zero-center using THIS dataset's stats
])
# EVAL transform: deterministic only -- no random augmentation, same Normalize.
eval_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_ds = datasets.MNIST(root="./data", train=True,  download=True, transform=train_tf)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,
                          num_workers=2, pin_memory=(device == "cuda"))
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False,   # eval: no shuffle
                          num_workers=2, pin_memory=(device == "cuda"))

imgs, labels = next(iter(train_loader))
print("MNIST batch:", imgs.shape, imgs.dtype, "| labels:", labels.shape)
# MNIST batch: torch.Size([128, 1, 28, 28]) torch.float32 | labels: torch.Size([128])

# ======================================================================
# 3) A CUSTOM collate_fn for VARIABLE-LENGTH data (e.g. sentences)
# ======================================================================
from torch.nn.utils.rnn import pad_sequence

class SeqDataset(Dataset):
    def __init__(self, seqs, labels):
        self.seqs, self.labels = seqs, labels
    def __len__(self):  return len(self.seqs)
    def __getitem__(self, i):
        return torch.tensor(self.seqs[i], dtype=torch.long), self.labels[i]

def pad_collate(batch):
    seqs, labels = zip(*batch)                 # lists of (seq_tensor, label)
    lengths = torch.tensor([len(s) for s in seqs])
    padded  = pad_sequence(seqs, batch_first=True, padding_value=0)  # (B, max_len)
    return padded, lengths, torch.tensor(labels)

seq_ds = SeqDataset([[1, 2, 3], [4, 5], [6, 7, 8, 9]], [0, 1, 0])
seq_loader = DataLoader(seq_ds, batch_size=3, collate_fn=pad_collate)
padded, lengths, lbls = next(iter(seq_loader))
print("padded sequences:", padded.shape, "| true lengths:", lengths.tolist())
# padded sequences: torch.Size([3, 4]) | true lengths: [3, 2, 4]

## Visualize the data & results

_Why does DataLoader parallelism matter? As we add worker processes, how does training throughput (samples/sec) change? Modelled on a small 2-conv-layer CNN (Convolutional Neural Network) whose GPU step takes ~20 ms per batch of 64, while one CPU loader needs ~115 ms of decode+augment per batch._

In [ ]:
import numpy as np

# Small illustrative model: a 2-conv-layer CNN. Per BATCH of 64 samples,
# the GPU forward+backward step takes ~20 ms (GPU-bound floor), while ONE
# CPU loader needs ~115 ms to decode+augment that batch. Extra workers split
# the CPU work; a small per-worker IPC overhead is added.
batch_size = 64
t_gpu      = 0.020     # 20 ms GPU step per batch  -> ceiling = 64/0.020 = 3200 samples/s
t_cpu_one  = 0.115     # 115 ms CPU decode+augment per batch with a single loader
overhead   = 0.0009    # ~0.9 ms extra per additional worker (process / IPC cost)

workers = [0, 1, 2, 3, 4, 6, 8, 12]
print(" w  throughput(samples/s)  per-batch(ms)")
for w in workers:
    eff = max(w, 1)                               # 0 workers == main process == 1 loader
    t_cpu  = t_cpu_one / eff + overhead * max(w - 1, 0)
    t_batch = max(t_gpu, t_cpu)                   # GPU and prefetch overlap -> the slower one wins
    thpt = batch_size / t_batch
    print(f"{w:2d}  {thpt:18.0f}  {t_batch*1000:12.1f}")
# w  throughput(samples/s)  per-batch(ms)
#  0                 557          115.0
#  1                 557          115.0
#  2                1096           58.4
#  3                1595           40.1
#  4                2035           31.4
#  6                2704           23.7
#  8                3096           20.7
# 12                3200           20.0   <- plateau at the 3200/s GPU ceiling

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your training is slow and nvidia-smi shows GPU utilisation bouncing between 0% and 100% instead of staying high. Your DataLoader uses the defaults: num_workers=0, pin_memory=False. What is happening and how do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize the symptom: spiky/low GPU utilisation means the GPU keeps finishing a batch and then waiting for the next one. — _With num_workers=0 the data is loaded in the main process, serially, in between training steps &mdash; so the GPU sits idle while the CPU decodes and augments._
- Raise num_workers to a small number (say 4) so worker processes prefetch the next batches in parallel while the GPU trains on the current one. — _Parallel prefetching overlaps CPU data work with GPU compute, so the next batch is ready the instant the GPU needs it._
- Set pin_memory=True and copy with .to(device, non_blocking=True). — _Page-locked host memory makes the CPU&rarr;GPU transfer faster and lets it overlap compute._
- Measure throughput (samples/sec) as you raise num_workers and stop at the plateau. — _Past the point where the GPU is saturated, more workers only add overhead &mdash; the plateau in the CODEVIZ chart._

**Answer:** The GPU is data-starved: with num_workers=0 every batch is loaded serially in the main process, so the GPU finishes a step and then idles while the CPU prepares the next batch &mdash; exactly the 0%&harr;100% sawtooth. Fix it by letting the loader prefetch in parallel: set num_workers to a few (e.g. 4), and add pin_memory=True with .to(device, non_blocking=True) for a faster, overlapping transfer. Raise num_workers while throughput climbs and stop at the plateau, since beyond GPU saturation extra workers just add overhead.

</details>

**Problem 2.** You copy your train DataLoader to make the validation one, leaving shuffle=True and the same transforms.Compose([RandomCrop(32, padding=4), RandomHorizontalFlip(), ToTensor(), Normalize(...)]). Your validation accuracy is noisy and looks lower than it should. What two things are wrong?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Spot shuffle=True on the validation loader. — _Validation should be a fixed, reproducible pass; shuffling it gains nothing and can break any per-sample bookkeeping. Shuffle the train loader only._
- Spot the augmentation (RandomCrop, RandomHorizontalFlip) in the validation transform. — _Random crops/flips on eval make each pass see different, distorted inputs, so the metric is noisy and not measuring true performance._
- Build a separate eval transform: deterministic only &mdash; e.g. ToTensor() + the same Normalize(...) (and resize/center-crop if needed), no randomness. — _Eval must be deterministic and undistorted so the metric is stable and reflects real accuracy._
- Set shuffle=False on the validation loader. — _A fixed order makes evaluation reproducible._

**Answer:** Two split-hygiene bugs. (1) shuffle=True on validation &mdash; only the train loader should shuffle; eval should be a fixed, reproducible pass (shuffle=False). (2) Augmentation on validation &mdash; RandomCrop and RandomHorizontalFlip distort each eval pass differently, making the metric noisy and unrepresentative. Give validation its own deterministic transform (ToTensor() + the same Normalize(...), plus resize/center-crop if needed) and keep the random augmentation for training only. Normalization must match across splits; randomness must not.

</details>

**Problem 3.** You write a custom Dataset for a classifier. __getitem__ returns (self.X[i], int(self.y[i])) where self.X is a numpy float64 array and self.y is a numpy array of class ids. Training crashes inside nn.CrossEntropyLoss / the model. What dtypes should __getitem__ return and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize the model expects torch.float32 input tensors, not a numpy float64 array or Python objects. — _Layer weights are float32; a float64 numpy array (or a non-tensor) causes a dtype mismatch error in the forward pass._
- Recognize nn.CrossEntropyLoss wants the target as a torch.long tensor of class indices (not one-hot, not float, not a Python int). — _CrossEntropyLoss indexes into the logits with integer class ids; it requires a long tensor of indices._
- Return (torch.tensor(self.X[i], dtype=torch.float32), torch.tensor(int(self.y[i]), dtype=torch.long)). — _Explicit casts give the input float32 and the target a scalar long, exactly what the model and loss expect; the default collate then stacks them into a clean batch._

**Answer:** __getitem__ must return real tensors with the right dtypes: the input as torch.float32 (the model's weights are float32, so a numpy float64 array or a Python object mismatches) and the classification target as a scalar torch.long class index (because nn.CrossEntropyLoss takes integer class indices, not one-hot vectors, floats, or Python ints). So: return torch.tensor(x, dtype=torch.float32), torch.tensor(int(y), dtype=torch.long). With the correct dtypes, the default collate_fn stacks them into a (B, ...) float input batch and a (B,) long target batch and the loss is happy.

</details>